In [ ]:
import os  # API keys are read from environment variables
from google import genai

# Replace with your actual API key
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])


In [ ]:
from llama_index.core.readers import SimpleDirectoryReader
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.llama_dataset.generator import RagDatasetGenerator
from llama_index.core.llms import CustomLLM, LLMMetadata
from llama_index.core.base.llms.types import CompletionResponse
from pydantic import Field, ConfigDict
from typing import Any
import time
import re
from google import genai

# Create a Gemini LLM wrapper for llama-index with rate limiting and retry logic
class GeminiLLM(CustomLLM):
    model_config = ConfigDict(extra="allow")
    
    client: Any = Field(default=None, exclude=True)
    model_name: str = Field(default="gemini-2.5-flash")
    
    def __init__(self, client, model_name: str = "gemini-2.5-flash", **kwargs):
        super().__init__(client=client, model_name=model_name, **kwargs)
        # Initialize rate limiting variables (not Pydantic fields)
        object.__setattr__(self, '_last_request_time', time.time())
        object.__setattr__(self, '_request_count', 0)
        object.__setattr__(self, '_rate_limit_window_start', time.time())
    
    def _rate_limit(self):
        """Enforce rate limiting: max 10 requests per minute for free tier"""
        current_time = time.time()
        
        # Get or initialize rate limiting attributes
        last_time = getattr(self, '_last_request_time', current_time)
        request_count = getattr(self, '_request_count', 0)
        window_start = getattr(self, '_rate_limit_window_start', current_time)
        
        # Reset counter if a minute has passed
        if current_time - window_start >= 60:
            request_count = 0
            window_start = current_time
        
        # If we've hit the limit, wait until the window resets
        if request_count >= 10:
            wait_time = 60 - (current_time - window_start)
            if wait_time > 0:
                print(f"⏳ Rate limit reached. Waiting {wait_time:.1f} seconds...")
                time.sleep(wait_time + 2)  # Add 2 second buffer
                request_count = 0
                window_start = time.time()
        
        # Ensure minimum 7 seconds between requests to stay well under the 10/min limit
        # This is more conservative than the theoretical 6 seconds to account for API variability
        time_since_last = current_time - last_time
        min_delay = 7.0  # 7 seconds = ~8.5 requests per minute, well under the 10/min limit
        if time_since_last < min_delay:
            sleep_time = min_delay - time_since_last
            if sleep_time > 0.1:  # Only print if significant wait
                print(f"⏱️  Waiting {sleep_time:.1f} seconds to respect rate limit...")
            time.sleep(sleep_time)
        
        # Update attributes
        object.__setattr__(self, '_last_request_time', time.time())
        object.__setattr__(self, '_request_count', request_count + 1)
        object.__setattr__(self, '_rate_limit_window_start', window_start)
    
    def _extract_retry_delay(self, error_str: str) -> float:
        """Extract retry delay from error message"""
        # Look for "Please retry in X.XXs" pattern
        match = re.search(r'Please retry in ([\d.]+)s', error_str)
        if match:
            delay = float(match.group(1))
            # Add a small buffer to the suggested delay
            return delay + 2.0
        return 30.0  # Default 30 seconds if not found (conservative default)
    
    @property
    def metadata(self) -> LLMMetadata:
        return LLMMetadata(
            model_name=self.model_name,
            context_window=32768,
            num_output=4096,
        )
    
    def complete(self, prompt: str, **kwargs) -> CompletionResponse:
        max_retries = 5
        retry_count = 0
        
        while retry_count < max_retries:
            try:
                # Enforce rate limiting
                self._rate_limit()
                
                response = self.client.models.generate_content(
                    model=self.model_name,
                    contents=[prompt]
                )
                text = response.text if hasattr(response, 'text') else str(response)
                return CompletionResponse(text=text)
                
            except Exception as e:
                error_str = str(e)
                
                # Check if it's a rate limit error (429)
                if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str or "quota" in error_str.lower():
                    retry_count += 1
                    if retry_count < max_retries:
                        # Extract retry delay from error message
                        delay = self._extract_retry_delay(error_str)
                        print(f"⚠️ Rate limit error. Retrying in {delay:.1f} seconds... (attempt {retry_count}/{max_retries})")
                        time.sleep(delay)
                        # Reset rate limit counters after waiting
                        object.__setattr__(self, '_request_count', 0)
                        object.__setattr__(self, '_rate_limit_window_start', time.time())
                        continue
                    else:
                        return CompletionResponse(text=f"Error: Rate limit exceeded after {max_retries} retries. Please try again later.")
                else:
                    # For other errors, return immediately
                    return CompletionResponse(text=f"Error: {error_str}")
        
        return CompletionResponse(text="Error: Max retries exceeded")
    
    def stream_complete(self, prompt: str, **kwargs):
        # For streaming, you can implement this if needed
        response = self.complete(prompt, **kwargs)
        yield response


In [ ]:
# Initialize Gemini LLM
gemini_llm = GeminiLLM(client=client, model_name="gemini-2.5-flash")

# Load table_talks.txt file
documents = SimpleDirectoryReader(input_files=["table_talks.txt"]).load_data()

# Parse documents into nodes
parser = SimpleNodeParser()
nodes = parser.get_nodes_from_documents(documents)

# Generate QA dataset
dataset_gen = RagDatasetGenerator(
    nodes=nodes,
    llm=gemini_llm,
    num_questions_per_chunk=3
)

dataset = dataset_gen.generate_dataset_from_nodes()

# Save questions and answers to a text file
output_file = "table_talks_qa_dataset.txt"
with open(output_file, "w", encoding="utf-8") as f:
    for i, example in enumerate(dataset.examples, 1):
        f.write(f"Question {i}:\n")
        f.write(f"{example.query}\n\n")
        f.write(f"Answer {i}:\n")
        f.write(f"{example.reference_answer}\n\n")
        f.write("-" * 80 + "\n\n")

print(f"✅ Generated {len(dataset.examples)} Q&A pairs")
print(f"✅ Saved to {output_file}")

# Also print first few examples to preview
print("\nPreview of first 3 examples:")
for i, example in enumerate(dataset.examples[:3], 1):
    print(f"\nQ{i}: {example.query}")
    print(f"A{i}: {example.reference_answer[:200]}..." if len(example.reference_answer) > 200 else f"A{i}: {example.reference_answer}")